# Fako Online - Full Pipeline Server (Kaggle)

**Bark TTS + SadTalker + Easy-Wav2Lip**

Models are loaded lazily (one at a time) to avoid GPU OOM.

### Instructions
1. Enable **GPU T4 x2** (Settings > Accelerator)
2. Add datasets: `kingtechie/bark-model`, `kingtechie/sadtalker-model`, `kingtechie/wav2lip-model`
3. Run all cells in order
4. The server will start on port 8000

In [ ]:
# Cell 1: Setup paths + discover SadTalker structure
import os, glob, sys

WORKING_DIR = "/kaggle/working/outputs"
os.makedirs(WORKING_DIR, exist_ok=True)

BARK_DIR = "/kaggle/input/bark-model"
SADTALKER_DIR = "/kaggle/input/sadtalker-model"
WAV2LIP_DIR = "/kaggle/input/wav2lip-model"

# Check for nested subdirectories inside sadtalker-model
if os.path.exists(os.path.join(SADTALKER_DIR, "SadTalker")):
    SADTALKER_DIR = os.path.join(SADTALKER_DIR, "SadTalker")
    print(f"Found nested SadTalker subdirectory")
elif os.path.exists(os.path.join(SADTALKER_DIR, "sadtalker")):
    SADTALKER_DIR = os.path.join(SADTALKER_DIR, "sadtalker")
    print(f"Found nested sadtalker subdirectory")

# Auto-discover where src/gradio_demo.py lives
sadtalker_gradio = glob.glob(f"{SADTALKER_DIR}/**/src/gradio_demo.py", recursive=True)
if sadtalker_gradio:
    SADTALKER_ROOT = os.path.dirname(os.path.dirname(sadtalker_gradio[0]))
    print(f"Found SadTalker root: {SADTALKER_ROOT}")
else:
    SADTALKER_ROOT = SADTALKER_DIR
    print(f"WARNING: src/gradio_demo.py not found, using {SADTALKER_DIR}")
    if os.path.exists(SADTALKER_DIR):
        for item in os.listdir(SADTALKER_DIR):
            print(f"  {item}")
    else:
        print(f"  Directory does not exist!")

# Auto-discover checkpoints and config
sadtalker_ckpt = glob.glob(f"{SADTALKER_ROOT}/**/checkpoints", recursive=True)
SADTALKER_CKPT = sadtalker_ckpt[0] if sadtalker_ckpt else f"{SADTALKER_ROOT}/checkpoints"

sadtalker_cfg = glob.glob(f"{SADTALKER_ROOT}/**/src/config", recursive=True)
SADTALKER_CONFIG = sadtalker_cfg[0] if sadtalker_cfg else f"{SADTALKER_ROOT}/src/config"

sadtalker_gfpgan = glob.glob(f"{SADTALKER_ROOT}/**/gfpgan/weights", recursive=True)
SADTALKER_GFPGAN = sadtalker_gfpgan[0] if sadtalker_gfpgan else f"{SADTALKER_ROOT}/gfpgan/weights"

# Check wav2lip checkpoint
wav2lip_ckpt = glob.glob(f"{WAV2LIP_DIR}/**/wav2lip_gan.pth", recursive=True)
WAV2LIP_CKPT = wav2lip_ckpt[0] if wav2lip_ckpt else f"{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth"

print(f"Bark: {BARK_DIR}")
print(f"SadTalker root: {SADTALKER_ROOT}")
print(f"SadTalker checkpoints: {SADTALKER_CKPT}")
print(f"SadTalker config: {SADTALKER_CONFIG}")
print(f"SadTalker GFPGAN: {SADTALKER_GFPGAN}")
print(f"Easy-Wav2Lip: {WAV2LIP_DIR}")
print(f"Wav2Lip checkpoint: {WAV2LIP_CKPT}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q fastapi uvicorn python-multipart
!pip install -q transformers scipy
!pip install -q gfpgan
!pip install -q librosa
!pip install -q pyngrok
print("Dependencies installed!")

In [ ]:
# Cell 3: Import core libraries
import torch
import numpy as np
import io
import base64
import subprocess
import sys
import gc
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Lazy model loading — only load what we need, when we need it
# Models start as None and get loaded on first API call, then moved to CPU after

_sadtalker_instance = None
_bark_processor = None
_bark_model = None

def get_sadtalker():
    global _sadtalker_instance
    if _sadtalker_instance is None:
        print("Loading SadTalker into GPU...")
        if SADTALKER_ROOT not in sys.path:
            sys.path.insert(0, SADTALKER_ROOT)
        # SadTalker relative imports require CWD set to its root
        original_cwd = os.getcwd()
        os.chdir(SADTALKER_ROOT)
        from src.gradio_demo import SadTalker
        _sadtalker_instance = SadTalker(
            checkpoint_path=SADTALKER_CKPT,
            config_path=SADTALKER_CONFIG,
            lazy_dir=SADTALKER_GFPGAN
        )
        os.chdir(original_cwd)
        print("SadTalker loaded!")
    return _sadtalker_instance

def get_bark():
    global _bark_processor, _bark_model
    if _bark_model is None:
        print("Loading Bark TTS into GPU...")
        from transformers import AutoProcessor, BarkModel
        _bark_processor = AutoProcessor.from_pretrained(BARK_DIR)
        _bark_model = BarkModel.from_pretrained(BARK_DIR)
        print("Bark TTS loaded!")
    return _bark_processor, _bark_model

def unload_sadtalker():
    global _sadtalker_instance
    if _sadtalker_instance is not None:
        del _sadtalker_instance
        _sadtalker_instance = None
        gc.collect()
        torch.cuda.empty_cache()
        print("SadTalker unloaded from GPU")

def unload_bark():
    global _bark_processor, _bark_model
    if _bark_model is not None:
        del _bark_model
        _bark_model = None
        _bark_processor = None
        gc.collect()
        torch.cuda.empty_cache()
        print("Bark unloaded from GPU")

print("Lazy loading functions ready!")
print("Models will load on first API call, then move to CPU after each use.")

In [ ]:
# Cell 5: Generation functions
import soundfile as sf

def generate_tts(text, voice_preset="v2/en_speaker_6"):
    processor, bark_model = get_bark()
    inputs = processor(text, voice_preset=voice_preset, return_tensors="pt")
    inputs = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in inputs.items()}
    bark_model.to("cuda")
    with torch.no_grad():
        audio_values = bark_model.generate(**inputs, do_sample=True)
    audio = audio_values.cpu().numpy().squeeze()
    bark_model.to("cpu")
    torch.cuda.empty_cache()
    output_path = f"{WORKING_DIR}/emotional_speech.wav"
    sf.write(output_path, audio, bark_model.generation_config.sample_rate)
    return output_path

def generate_avatar(image_path, audio_path, result_filename="talking-head.mp4"):
    sadtalker = get_sadtalker()
    result = sadtalker.test(
        image_path=image_path,
        audio_path=audio_path,
        result_dir=WORKING_DIR,
        still_mode=False,
        use_enhancer=True,
        batch_size=2,
        size=256,
        pose_style=0
    )
    # Move SadTalker to CPU after use
    unload_sadtalker()
    return result["mp4_path"] if isinstance(result, dict) else result

def run_wav2lip(video_path, audio_path):
    output_path = f"{WORKING_DIR}/refined_output.mp4"
    cmd = [
        "python", f"{WAV2LIP_DIR}/inference.py",
        "--checkpoint_path", WAV2LIP_CKPT,
        "--face", video_path,
        "--audio", audio_path,
        "--outfile", output_path,
        "--nosmooth"
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    return output_path

print("Generation functions defined!")

In [ ]:
# Cell 6: FastAPI server
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title="Fako Online - Full Pipeline API")

@app.get("/health")
async def health():
    return {"status": "ok", "models": ["bark", "sadtalker", "wav2lip"]}

@app.post("/generate-tts")
async def api_generate_tts(text: str = Form(...), voice_preset: str = Form("v2/en_speaker_6")):
    try:
        audio_path = generate_tts(text, voice_preset)
        return FileResponse(audio_path, media_type="audio/wav")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/generate-avatar")
async def api_generate_avatar(image: UploadFile = File(...), audio: UploadFile = File(...)):
    try:
        image_path = f"{WORKING_DIR}/{image.filename}"
        audio_path = f"{WORKING_DIR}/{audio.filename}"
        with open(image_path, "wb") as f:
            f.write(await image.read())
        with open(audio_path, "wb") as f:
            f.write(await audio.read())
        video_path = generate_avatar(image_path, audio_path)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/generate-full")
async def api_generate_full(
    image: UploadFile = File(...),
    text: str = Form(...),
    voice_preset: str = Form("v2/en_speaker_6"),
    refine_lips: bool = Form(True)
):
    try:
        image_path = f"{WORKING_DIR}/{image.filename}"
        with open(image_path, "wb") as f:
            f.write(await image.read())
        audio_path = generate_tts(text, voice_preset)
        # Move Bark to CPU before loading SadTalker
        unload_bark()
        video_path = generate_avatar(image_path, audio_path)
        if refine_lips:
            video_path = run_wav2lip(video_path, audio_path)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

print("FastAPI server defined!")

In [ ]:
# Cell 7: Start server (threaded to avoid asyncio conflict with Jupyter)
import uvicorn
import threading
import time
from pyngrok import ngrok

ngrok.set_auth_token("3JbM9BB0RTMJmMJtnx3EVCXlAoY_88KRTp3xpt6SGnX5ELFnS")

public_url = ngrok.connect(8000)
print(f"\n=== Server running ===")
print(f"Public URL: {public_url}", flush=True)
print(f"Use this URL in your local .env as KAGGLE_API_URL", flush=True)

# Save URL to file so it can be retrieved via 'kaggle kernels output'
with open("/kaggle/working/ngrok_url.txt", "w") as f:
    f.write(str(public_url))
print(f"URL saved to /kaggle/working/ngrok_url.txt", flush=True)

print(f"\nStarting FastAPI server on port 8000...", flush=True)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Keep alive for 2 hours (Kaggle max is ~9 hours)
for i in range(120):
    time.sleep(60)
print("Server stopped after 2 hours.")